# rStar-Math Model Comparison

This notebook demonstrates the performance comparison between different Language Models (LLMs) with and without rStar-Math enhancement.

In [ ]:
import os
import json
from typing import Dict, List
import pandas as pd
import plotly.express as px
from src.core.mcts import MCTS
from src.core.ppm import ProcessPreferenceModel
from src.models.model_interface import ModelFactory

## Setup Models

First, let's set up our LLMs with API keys.

In [ ]:
# Load API keys from environment variables
api_keys = {
    'openai': os.getenv('OPENAI_API_KEY'),
    'anthropic': os.getenv('ANTHROPIC_API_KEY'),
    'mistral': os.getenv('MISTRAL_API_KEY'),
    'groq': os.getenv('GROQ_API_KEY'),
    'gemini': os.getenv('GEMINI_API_KEY')
}

# Initialize models
models = {}
for model_name, api_key in api_keys.items():
    if api_key:
        models[model_name] = ModelFactory.create_model(
            model_name,
            api_key,
            'config/default.json'
        )

## Initialize rStar-Math Components

In [ ]:
mcts = MCTS.from_config_file('config/default.json')
ppm = ProcessPreferenceModel.from_config_file('config/default.json')

## Test Problems

Let's define some test problems of varying difficulty.

In [ ]:
test_problems = [
    "What is 2 + 2?",  # Simple arithmetic
    "Solve for x: 2x + 3 = 7",  # Basic algebra
    "Find the derivative of f(x) = x^2 + 3x",  # Calculus
    "In a group of 30 people, 40% are men. How many women are there?",  # Word problem
    "Prove that the square root of 2 is irrational"  # Mathematical proof
]

## Compare Model Performance

In [ ]:
def solve_problem(model, problem: str, use_rstar: bool = False) -> Dict:
    """Solve a problem with or without rStar-Math."""
    if use_rstar:
        action, trajectory = mcts.search(problem)
        solution_steps = [step['state'] for step in trajectory]
        score = sum(ppm.evaluate_step(step, model) for step in solution_steps) / len(solution_steps)
    else:
        solution = model.generate_response(problem)
        solution_steps = [solution]
        score = model.evaluate_reasoning(problem, solution_steps)
        
    return {
        'solution': '\n'.join(solution_steps),
        'score': score
    }

# Collect results
results = []
for problem in test_problems:
    for model_name, model in models.items():
        # Without rStar-Math
        direct_result = solve_problem(model, problem, use_rstar=False)
        results.append({
            'problem': problem,
            'model': model_name,
            'method': 'Direct',
            'score': direct_result['score']
        })
        
        # With rStar-Math
        rstar_result = solve_problem(model, problem, use_rstar=True)
        results.append({
            'problem': problem,
            'model': model_name,
            'method': 'rStar-Math',
            'score': rstar_result['score']
        })

# Create DataFrame
df = pd.DataFrame(results)

## Visualize Results

In [ ]:
# Overall comparison
fig = px.box(df, x='model', y='score', color='method',
             title='Model Performance Comparison')
fig.show()

# Problem-specific comparison
fig = px.bar(df, x='model', y='score', color='method',
             facet_row='problem', barmode='group',
             title='Model Performance by Problem Type')
fig.show()

# Calculate improvement statistics
improvements = df[df['method'] == 'rStar-Math']['score'].mean() - \
              df[df['method'] == 'Direct']['score'].mean()
print(f"Average improvement with rStar-Math: {improvements:.2%}")

## Analyze Solution Steps

Let's look at detailed solution steps for a specific problem.

In [ ]:
def analyze_solution(model, problem: str) -> None:
    """Compare direct vs rStar-Math solutions."""
    print(f"Problem: {problem}\n")
    
    # Direct solution
    print("Direct Solution:")
    direct_result = solve_problem(model, problem, use_rstar=False)
    print(direct_result['solution'])
    print(f"Confidence Score: {direct_result['score']:.2f}\n")
    
    # rStar-Math solution
    print("rStar-Math Enhanced Solution:")
    rstar_result = solve_problem(model, problem, use_rstar=True)
    print(rstar_result['solution'])
    print(f"Confidence Score: {rstar_result['score']:.2f}")

# Analyze a complex problem
complex_problem = test_problems[-1]  # Proof problem
model = models['openai']  # Use GPT-4 for demonstration
analyze_solution(model, complex_problem)